# 301 · Untrusted input experiment

Companion to [Untrusted input](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/301/untrusted-input/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/301/untrusted_input.ipynb)

Hard **size / depth / cardinality** limits around JSON; never native-deserialize untrusted bytes. No live RCE demos.

> **Honesty:** suite speed ≠ adversarial robustness.


## Resource limits as part of the parser boundary

Cap bytes, nesting, and collection size so DoS shapes fail closed—ideally before paying full materialization cost.


In [ ]:
import json
from typing import Any


class Limits:
    max_bytes = 10_000
    max_depth = 8
    max_collection = 100


def check_bytes(raw: bytes, lim: Limits = Limits()) -> None:
    if len(raw) > lim.max_bytes:
        raise ValueError(f"body too large: {len(raw)} > {lim.max_bytes}")


def check_depth(obj: Any, lim: Limits = Limits(), depth: int = 0) -> None:
    if depth > lim.max_depth:
        raise ValueError(f"nesting too deep: {depth}")
    if isinstance(obj, dict):
        if len(obj) > lim.max_collection:
            raise ValueError("too many map keys")
        for v in obj.values():
            check_depth(v, lim, depth + 1)
    elif isinstance(obj, list):
        if len(obj) > lim.max_collection:
            raise ValueError("too many array elements")
        for v in obj:
            check_depth(v, lim, depth + 1)


def parse_untrusted_json(raw: bytes, lim: Limits = Limits()) -> Any:
    check_bytes(raw, lim)
    obj = json.loads(raw)
    check_depth(obj, lim)
    return obj



## Benign vs hostile-shaped JSON

One good parse; depth / wide-map / oversized body each raise `ValueError`.


In [ ]:
ok = json.dumps({"user": "a", "items": [1, 2, 3]}).encode()
print("OK parse:", parse_untrusted_json(ok))

# depth bomb
depth = {"x": 0}
cur = depth
for _ in range(20):
    cur["n"] = {}
    cur = cur["n"]
deep = json.dumps(depth).encode()
try:
    parse_untrusted_json(deep)
    raise AssertionError("expected depth failure")
except ValueError as e:
    print("OK depth rejected:", e)

# cardinality bomb
wide = json.dumps({"k" + str(i): i for i in range(500)}).encode()
try:
    parse_untrusted_json(wide)
    raise AssertionError("expected cardinality failure")
except ValueError as e:
    print("OK cardinality rejected:", e)

# size bomb
big = b"{" + b'"a":"' + b"x" * 20_000 + b'"}'
try:
    parse_untrusted_json(big)
    raise AssertionError("expected size failure")
except ValueError as e:
    print("OK size rejected:", e)



## Never native-deserialize untrusted bytes

Default boundary path is portable JSON+limits. Pickle only if explicitly allowed for trusted demos.


In [ ]:
import pickle

def load_boundary(raw: bytes, allow_pickle: bool = False):
    if allow_pickle:
        # only for fully trusted same-process demos
        return pickle.loads(raw)
    # production default at untrusted boundary
    return parse_untrusted_json(raw)


trusted_local = pickle.dumps({"cache": True})
try:
    load_boundary(trusted_local, allow_pickle=False)
except Exception as e:
    print("OK pickle blocked at boundary:", type(e).__name__, e)

print("OK portable path:", load_boundary(ok, allow_pickle=False))



## Takeaways

Assume hostile input. Limits belong at the boundary. Related: [Trust boundaries](./trust_boundaries.ipynb).
